# P3

## Dataset as a product

In [8]:
import heapq
from collections import Counter
import json
import numpy as np

class Node:
	def __init__(self, symbol=None, frequency=None, order=0):
		self.symbol = symbol
		self.frequency = frequency
		self.order = order
		self.left = None
		self.right = None
	def __lt__(self, other):
		if self.frequency != other.frequency:
			return self.frequency < other.frequency
		return self.order < other.order

def build_huffman_tree(counts):
	heap = []

	for order, (sym, f) in enumerate(sorted(counts.items())):
		heapq.heappush(heap, Node(sym, f, order))

	order = len(heap)

	while len(heap) > 1:
		left = heapq.heappop(heap)
		right = heapq.heappop(heap)
		merged = Node(frequency=left.frequency + right.frequency, order = order)
		merged.left = left
		merged.right = right
		heapq.heappush(heap, merged)

	return heap[0]

def get_code_lengths_from_tree(root):
	lengths = {}

	def walk(node, depth=0):
		if node.symbol is not None:
			lengths[node.symbol] = depth
		else:
			walk(node.left, depth + 1)
			walk(node.right, depth + 1)

	walk(root)

	if len(lengths) == 1:
		lengths[next(iter(lengths))] = 1

	return lengths

def huffman_lengths(counts):
	if not counts:
		return {}

	root = build_huffman_tree(counts)
	return get_code_lengths_from_tree(root)

def canonical_codes(lengths):
	present = sorted(lengths, key=lambda s: (lengths[s], s))
	codes = {}
	code = 0
	prev_len = 0

	for s in present:
		L = lengths[s]
		code <<= (L - prev_len)
		codes[s] = f"{code:0{L}b}"
		code += 1
		prev_len = L

	return codes

MAGIC_BYTES = b"HC\x01"  # used by a series of bytes
MAGIC_ARRAY = b"HC\x02H" # used by numpy arrays

BASE_HEADER_BYTES = 13

def compress_bytes(data):
	if not isinstance(data, (bytes, bytearray)):
		raise TypeError("compress_bytes expects bytes")

	data = bytes(data)

	if not data:
		return MAGIC_BYTES + (0).to_bytes(8, "big") + (0).to_bytes(2, "big")

	freq = Counter(data)
	lengths = huffman_lengths(freq)
	codes = canonical_codes(lengths)

	present = sorted(freq)
	header = bytearray(MAGIC_BYTES)             
	header += len(data).to_bytes(8, "big")    
	header += len(present).to_bytes(2, "big")

	for s in present:
		header.append(s)
		header.append(lengths[s])
		
	bitstring = "".join(codes[b] for b in data)
	nbits = len(bitstring)
	payload = int(bitstring + "0" * (-nbits % 8), 2).to_bytes((nbits + 7) // 8, "big") if nbits else b""
	
	return bytes(header) + payload

def parse_header(compressed):
	compressed = bytes(compressed)

	if len(compressed) < BASE_HEADER_BYTES or compressed[:3] != MAGIC_BYTES:
		raise ValueError("bad header: wrong magic or stream too short")

	orig_len = int.from_bytes(compressed[3:11], "big")
	nums_syms = int.from_bytes(compressed[11:13], "big")

	pos = BASE_HEADER_BYTES
	present = []
	lengths = {}

	for _ in range(nums_syms):
		s = compressed[pos]
		l = compressed[pos + 1]

		if l == 0:
			raise ValueError(f"bad code length 0 for symbol {s}")

		if s in lengths:
			raise ValueError(f"duplicate symbol {s} in table")

		present.append(s)
		lengths[s] = l
		pos += 2

	return orig_len, present, lengths, pos

def decompress_bytes(compressed):
	orig_len, _, lengths, pos = parse_header(compressed)

	if orig_len == 0:
		return b""

	codes = canonical_codes(lengths)
	decoded_table = {code: sym for sym, code in codes.items()}

	out = bytearray()
	current = ""

	for byte in compressed[pos:]:
		for i in range(7, -1, -1):
			current += "1" if byte & (1 << i) else "0"

			if current in decoded_table:
				out.append(decoded_table[current])
				current = ""

				if len(out) == orig_len:
					return bytes(out)

	raise ValueError("truncated bitstream")

from functools import lru_cache

PREDICTORS = (
    "none",
    "horizontal_delta",
    "vertical_delta",
    "spiral_delta",
    "auto",
)

@lru_cache(maxsize=None)
def spiral_order(h, w):
    top, bottom, left, right = 0, h - 1, 0, w - 1
    idx = []

    while top <= bottom and left <= right:
        for x in range(left, right + 1):
            idx.append(top * w + x)

        for y in range(top + 1, bottom + 1):
            idx.append(y * w + right)

        if bottom > top:
            for x in range(right - 1, left - 1, -1):
                idx.append(bottom * w + x)

        if right > left and bottom > top:
            for y in range(bottom - 1, top, -1):
                idx.append(y * w + left)

        top += 1
        bottom -= 1
        left += 1
        right -= 1

    assert len(idx) == h * w
    order = np.array(idx, dtype=np.int64)
    order.setflags(write=False)
    return order


def is_valid(arr):
    return (
        isinstance(arr, np.ndarray)
        and arr.dtype == np.uint8
        and arr.ndim == 3
        and arr.shape[2] == 3
    )


def preprocess_delta(arr, direction="horizontal"):
    if arr.size == 0:
        return arr.copy()

    if not is_valid(arr):
        raise ValueError(
            f"Delta predictor needs uint8 HxWx3, "
            f"got {arr.dtype} {arr.shape}"
        )

    if direction == "horizontal":
        residual = np.empty_like(arr)
        residual[:, 0, :] = arr[:, 0, :]
        residual[:, 1:, :] = (
            arr[:, 1:, :].astype(np.int16)
            - arr[:, :-1, :].astype(np.int16)
        ) % 256

    elif direction == "vertical":
        residual = np.empty_like(arr)
        residual[0, :, :] = arr[0, :, :]
        residual[1:, :, :] = (
            arr[1:, :, :].astype(np.int16)
            - arr[:-1, :, :].astype(np.int16)
        ) % 256

    elif direction == "spiral":
        h, w, _ = arr.shape
        order = spiral_order(h, w)
        along = arr.reshape(-1, 3)[order]

        residual = np.empty_like(along)
        residual[0] = along[0]
        residual[1:] = (
            along[1:].astype(np.int16)
            - along[:-1].astype(np.int16)
        ) % 256

        out = np.empty((h * w, 3), dtype=np.uint8)
        out[order] = residual
        residual = out.reshape(h, w, 3)

    else:
        raise ValueError(f"Unknown predictor direction: {direction}")

    return residual.astype(np.uint8)


def postprocess_delta(residual, direction="horizontal"):
    if residual.size == 0:
        return residual.astype(np.uint8)

    recon = np.empty_like(residual)

    if direction == "horizontal":
        recon[:, 0, :] = residual[:, 0, :]
        for x in range(1, residual.shape[1]):
            recon[:, x, :] = (
                residual[:, x, :].astype(np.int16)
                + recon[:, x - 1, :].astype(np.int16)
            ) % 256

    elif direction == "vertical":
        recon[0, :, :] = residual[0, :, :]
        for y in range(1, residual.shape[0]):
            recon[y, :, :] = (
                residual[y, :, :].astype(np.int16)
                + recon[y - 1, :, :].astype(np.int16)
            ) % 256

    elif direction == "spiral":
        h, w, _ = residual.shape
        order = spiral_order(h, w)
        along = residual.reshape(-1, 3)[order]

        rec = np.cumsum(along.astype(np.int64), axis=0) % 256

        out = np.empty((h * w, 3), dtype=np.uint8)
        out[order] = rec
        recon = out.reshape(h, w, 3)

    else:
        raise ValueError(f"Unknown predictor direction: {direction}")

    return recon.astype(np.uint8)


def encode_with_predictor(arr, predictor):
    if predictor != "none" and not is_valid(arr):
        predictor = "none"

    if predictor == "horizontal_delta":
        encoded_arr = preprocess_delta(arr, "horizontal")
    elif predictor == "vertical_delta":
        encoded_arr = preprocess_delta(arr, "vertical")
    elif predictor == "spiral_delta":
        encoded_arr = preprocess_delta(arr, "spiral")
    else:
        encoded_arr = arr

    meta = {
        "shape": list(arr.shape),
        "dtype": str(arr.dtype),
        "predictor": predictor,
    }
    meta_bytes = json.dumps(
        meta, separators=(",", ":")
    ).encode("utf-8")

    if len(meta_bytes) > 0xFFFFFFFF:
        raise ValueError("Array metadata is too large")

    raw_bytes = np.ascontiguousarray(encoded_arr).tobytes()
    payload = compress_bytes(raw_bytes)

    return (
        MAGIC_ARRAY
        + len(meta_bytes).to_bytes(4, "big")
        + meta_bytes
        + payload
    )


def compress_array(arr, settings=None):
    if not isinstance(arr, np.ndarray):
        raise TypeError("compress_array expects a np.ndarray")

    settings = settings or {}
    predictor = settings.get("predictor", "auto")

    if predictor not in PREDICTORS:
        raise ValueError(f"Unknown predictor: {predictor}. Valid: {PREDICTORS}")

    if predictor == "auto":
        if not is_valid(arr):
            return encode_with_predictor(arr, "none")

        candidates = (
            encode_with_predictor(arr, "none"),
            encode_with_predictor(arr, "spiral_delta"),
            encode_with_predictor(arr, "vertical_delta"),
            encode_with_predictor(arr, "horizontal_delta"),
        )

        return min(candidates, key=len)

    return encode_with_predictor(arr, predictor)


def decompress_array(compressed):
    if len(compressed) < 4:
        raise ValueError("Truncated array header")

    meta_len = int.from_bytes(compressed[:4], "big")
    meta_end = 4 + meta_len

    if meta_end > len(compressed):
        raise ValueError("Truncated array metadata")

    try:
        meta = json.loads(compressed[4:meta_end].decode("utf-8"))
        shape = tuple(meta["shape"])
        dtype = np.dtype(meta["dtype"])
        predictor = meta["predictor"]
    except (UnicodeDecodeError, json.JSONDecodeError, KeyError,
            TypeError, ValueError) as exc:
        raise ValueError("Invalid array metadata") from exc

    if predictor not in PREDICTORS or predictor == "auto":
        raise ValueError(f"Invalid stored predictor: {predictor}")

    if any(not isinstance(d, int) or d < 0 for d in shape):
        raise ValueError("Invalid array shape")

    raw_bytes = decompress_bytes(compressed[meta_end:])

    expected_nbytes = int(np.prod(shape, dtype=object)) * dtype.itemsize
    if len(raw_bytes) != expected_nbytes:
        raise ValueError(
            f"Decoded byte count mismatch: expected {expected_nbytes}, "
            f"got {len(raw_bytes)}"
        )

    arr = np.frombuffer(raw_bytes, dtype=dtype).copy().reshape(shape)

    if predictor != "none":
        if not is_valid(arr):
            raise ValueError("Stored predictor is incompatible with array")

        direction = {
            "horizontal_delta": "horizontal",
            "vertical_delta": "vertical",
            "spiral_delta": "spiral",
        }[predictor]

        arr = postprocess_delta(arr, direction)

    return arr

def compress(data, settings=None):
	if isinstance(data, np.ndarray):
		return compress_array(data, settings)
	return compress_bytes(data)

def decompress(compressed):
	if compressed[:len(MAGIC_ARRAY)] == MAGIC_ARRAY:
		return decompress_array(compressed[4:])
	if compressed[:len(MAGIC_BYTES)] == MAGIC_BYTES:
		return decompress_bytes(compressed)
	raise ValueError("unknown format: " + repr(compressed[:4]))

from pathlib import Path
RESOURCE_DIR_KIRMIZI = Path("dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Kirmizi_Pistachio")

def load_images(folder: Path, limit=None):
    records = []
    for path in sorted(folder.rglob("*")):
        if path.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        if limit is not None and len(records) >= limit:
            break
        with Image.open(path) as im:
            arr = np.asarray(im.convert("RGB"))
        records.append({"path": path, "arr": arr})
    return records

In [ ]:
import hashlib, json, logging, os, random, time, tracemalloc
from pathlib import Path
import numpy as np

CONFIG = {
    "seed": 42,
    "raw_dir": Path("dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Kirmizi_Pistachio"),
    "out_dir": Path("data/processed"),
    "manifest_path": Path("data/manifest.json"),
    "expected_shape": (600, 600, 3),
    "expected_dtype": np.dtype("uint8"),
    "valid_min": 0, "valid_max": 255,
    "predictor": "auto",
}
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"])
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("p3")

In [ ]:
class SchemaError(ValueError): pass

def validate(arr):
    """Contract: shape, dtype, valid range."""
    if not isinstance(arr, np.ndarray):
        raise SchemaError(f"Expected ndarray, got {type(arr).__name__}")
    if arr.shape != CONFIG["expected_shape"]:
        raise SchemaError(f"Invalid shape {arr.shape}, expected {CONFIG['expected_shape']}")
    if arr.dtype != CONFIG["expected_dtype"]:
        raise SchemaError(f"Invalid dtype {arr.dtype}, expected {CONFIG['expected_dtype']}")
    if (arr < CONFIG["valid_min"]).any() or (arr > CONFIG["valid_max"]).any():
        raise SchemaError(f"Values outside [{CONFIG['valid_min']}, {CONFIG['valid_max']}]")

def load_raw(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")
    arr = np.load(path)
    validate(arr)
    return arr

def write_atomic(path, data: bytes):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as fh:
        fh.write(data); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)   # half-written files never look complete